In [2]:
!pip install --upgrade langchain langchain-core langchain-community langchain-ollama langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.7 MB/s eta 0:00:00


In [5]:
import os

os.environ["OLLAMA_BASE_URL"] = "https://reentry-subject-mannish.ngrok-free.dev"

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def main():
    # 1. Load your local document (e.g., a PDF)
    # Replace with the path to your own local file
    pdf_path = r"/content/GRU.pdf"
    if not os.path.exists(pdf_path):
        print(f"Please place a valid PDF at '{pdf_path}' to test the script.")
        return

    print("Loading and splitting document...")
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    # 2. Chunk the document text
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    # 3. Initialize Ollama Local Embeddings
    embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="https://reentry-subject-mannish.ngrok-free.dev")

    # 4. Embed chunks and store them in a local Chroma vector database
    print("Generating embeddings and building vector store...")
    vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 5. Initialize the Local Ollama Chat LLM
    llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="https://reentry-subject-mannish.ngrok-free.dev")

    # 6. Define the RAG System Prompt
    system_prompt = (
        "You are an assistant for question-answering tasks.\n"
        "Use the following pieces of retrieved context to answer the question.\n"
        "If you don't know the answer, say that you don't know.\n\n"
        "Context:\n{context}"
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{question}"),
    ])

    # 7. Construct the LangChain Expression Language (LCEL) Chain
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # 8. Query the local RAG system
    query = "What are the main key findings or summaries in this document?"
    print(f"\nAsking local LLM: '{query}'\n" + "-"*50)

    response = rag_chain.invoke(query)
    print(response)

if __name__ == "__main__":
    main()


Loading and splitting document...
Generating embeddings and building vector store...

Asking local LLM: 'What are the main key findings or summaries in this document?'
--------------------------------------------------
Based on the provided context, the main key findings or summaries in this document are:

1. The document is a tutorial paper on Recurrent Neural Networks (RNN), Long Short-Term Memory (LSTM), and its variants.
2. It covers various topics, including dynamical systems, backpropagation through time, LSTM gates and cells, history and variants of LSTM, the Gated Recurrent Unit (GRU) cell, bidirectional RNN, bidirectional LSTM, and the ELMo network.
3. The document discusses the LSTM cell, which has four gates: input gate, forget gate, output gate, and cell gate. The input gate, forget gate, and output gate are explained in detail.
4. The document also discusses the GRU cell, which is a simplification of the LSTM cell. The GRU cell has three gates: reset gate, update gate, and